In [1]:
import numpy as np
import pickle
import pyvista as pv
from matplotlib.colors import ListedColormap

#pv.global_theme.colorbar_vertical.width = 0.05

In [2]:
snap = 84
with open(f"thres30ini_snap{snap:02d}_clump_core.pickle", "rb") as f:
    data = pickle.load(f)

### Find peak volume density in simulations

In [3]:
for index in range(1, 10):
    print("Peak density:", data["clump"][index].max() * 16.96)

Peak density: 2.2078044e+07
Peak density: 290674.75
Peak density: 122115.05
Peak density: 65371.4
Peak density: 35747.496
Peak density: 22525.6
Peak density: 54251.234
Peak density: 27478.629
Peak density: 28224.81


### Load simulation data

In [4]:
clump_index = 2 # choose 2 or 4 to reproduce the visualization in the paper

print(data["clump"][clump_index].shape, "\n",
      data["clump"].keys(),   "\n",
      data.keys())

(102, 126, 117) 
 dict_keys([np.uint64(1), np.uint64(2), np.uint64(3), np.uint64(4), np.uint64(5), np.uint64(6), np.uint64(7), np.uint64(8), np.uint64(9)]) 
 dict_keys(['clump', 'clump_vx', 'clump_vy', 'clump_vz', 'clump_Bx', 'clump_By', 'clump_Bz', 'clump_mask', 'peak_core_mask'])


### Visualize volume density distribution

In [5]:
# Load density data
core = data["clump"][clump_index] * 16.96 * data["clump_mask"][clump_index]

# Create the grid for holding the density array
grid = pv.ImageData(dimensions=core.shape)
grid.origin = (0, 0, 0)
grid.spacing = (1, 1, 1)
grid.point_data["density"] = core.flatten(order="F")

# Create a custom scalar bar (colorbar)
sargs = {
    # Title of the colorbar
    "title": "Gas density (particles / cc)",
    # Units displayed after the title
    "title_font_size": 34,
    "label_font_size": 34,

    # Format of the labels
    "fmt": "%.1e",  # Scientific notation
    
    # Position and size
    "height": 0.5,
    "width": 0.08,  # Controls the width of the colorbar 
    "vertical": True,
    "position_x": 0.85,
    "position_y": 0.2,
    
    # Units label
    "n_labels": 5,
    "shadow": True,

    "interactive": True
}

# Create plotter
plotter = pv.Plotter(window_size=[1440, 1080])

# Customize transfer function for opacity
opacity_tf = [0.0, 0.05, 0.1, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# Add volume with custom colorbar title
plotter.add_volume(
    grid, 
    cmap = "Greys",
    clim = [0, 6.3e4], #1e5
    #opacity = "linear",
    opacity = opacity_tf,
    scalar_bar_args = sargs,
    show_scalar_bar = False
)

<Volume(0x000001EC99330150) at 0x000001EC92FA5240>

### Visualize magnetic fields

In [6]:
# magnetic fields
Bx = data["clump_Bx"][clump_index]
By = data["clump_By"][clump_index]
Bz = data["clump_Bz"][clump_index]

# Calculate field magnitude
B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)

# Add B-field components to the grid
grid.point_data["magnetic_field"] = np.column_stack([
    Bx.flatten(order="F"),  # x-component
    By.flatten(order="F"),  # y-component
    Bz.flatten(order="F")   # z-component
])
grid.point_data["B_magnitude"] = B_mag.flatten(order="F")


# Create seed points for the streamlines
shape = grid.dimensions
x_dim, y_dim, z_dim = shape[0], shape[1], shape[2]
sampling_points = 28 # determines the number of field lines plotted;

# Create a uniform regular grid for sampling
x_sample = np.linspace(0, x_dim-1, sampling_points)
y_sample = np.linspace(0, y_dim-1, sampling_points)
z_sample = np.linspace(0, z_dim-1, sampling_points)

threshold = np.max(core) * 0.05  # adjust as needed for plotting

# Create seed points based on density distribution
seed_points = []
for x in x_sample:
    for y in y_sample:
        for z in z_sample:
            # Convert to integer indices for density lookup
            xi, yi, zi = int(x), int(y), int(z)
            
            # Check if we're within bounds
            if (0 <= xi < core.shape[0] and 
                0 <= yi < core.shape[1] and 
                0 <= zi < core.shape[2]):
                
                # Get density at this location
                local_density = core[xi, yi, zi]
                
                # Probability of placing a seed point based on density
                if local_density > threshold:
                    # Higher density regions: always place a seed
                    seed_points.append([x, y, z])
                elif local_density > 0:
                    # Low density regions: place seed with probability proportional to density
                    prob = local_density / threshold
                    if np.random.random() < prob:
                        seed_points.append([x, y, z])
                # Zero density regions: no seeds (implicit)
            
# Convert to NumPy array and then to PolyData
seed_points = np.array(seed_points)
seeds = pv.PolyData(seed_points)

# use streamlines_from_source directly with correct parameters
streamlines = grid.streamlines_from_source(
    seeds,  # This is the first positional argument - the source
    "magnetic_field",  # This is the second positional argument - the vectors
    integration_direction="both",
    initial_step_length=0.5,
    max_steps=1000,
    terminal_speed=1e-8
)

# Filter streamlines based on maximum density they pass through
def filter_streamlines_by_max_density(streamlines, density_grid):
    """
    Filter streamlines into sets A (max density < 1e4) and C (max density > 5e4).
    Ignore set B (1e4 < max density < 5e4).
    """
    low_threshold = 1e4
    high_threshold = 5e4
    
    # Get streamline points
    points = streamlines.points
    print(f"Analyzing {len(points)} streamline points...")
    
    # Sample density values at streamline points
    sampled_densities = []
    for point in points:
        # Convert to grid indices
        x_idx = int(np.clip(np.round(point[0]), 0, core.shape[0]-1))
        y_idx = int(np.clip(np.round(point[1]), 0, core.shape[1]-1))
        z_idx = int(np.clip(np.round(point[2]), 0, core.shape[2]-1))
        sampled_densities.append(core[x_idx, y_idx, z_idx])
    
    sampled_densities = np.array(sampled_densities)
    
    # Parse streamline connectivity and filter
    lines = streamlines.lines
    line_idx = 0
    filtered_streamlines_data = []
    filtered_classifications = []
    
    set_a_count = 0
    set_b_count = 0
    set_c_count = 0
    
    while line_idx < len(lines):
        n_points = lines[line_idx]
        line_idx += 1
        
        # Get indices for this streamline
        line_point_indices = lines[line_idx:line_idx + n_points]
        line_densities = sampled_densities[line_point_indices]
        
        # Find maximum density for this streamline
        max_density = np.max(line_densities)
        
        # Classify streamline based on maximum density
        if max_density < low_threshold:
            # Set A: max density < 1e4
            streamline_points = points[line_point_indices]
            filtered_streamlines_data.append(streamline_points)
            filtered_classifications.append(0.0)  # Blue for set A
            set_a_count += 1
        elif max_density > high_threshold:
            # Set C: max density > 1.5e5
            streamline_points = points[line_point_indices]
            filtered_streamlines_data.append(streamline_points)
            filtered_classifications.append(1.0)  # Red for set C
            set_c_count += 1
        else:
            # Set B: 1e4 < max density < 1.5e5 - ignore
            set_b_count += 1
        
        line_idx += n_points
    
    print(f"Set A (max density < 1e4): {set_a_count} streamlines")
    print(f"Set B (1e4 < max density < 1.5e5): {set_b_count} streamlines - ignored")
    print(f"Set C (max density > 1.5e5): {set_c_count} streamlines")
    print(f"Total kept: {len(filtered_streamlines_data)} streamlines")
    
    if len(filtered_streamlines_data) == 0:
        print("No streamlines found in sets A or C!")
        return None, None
    
    # Create new streamlines object with filtered data
    filtered_points = np.vstack(filtered_streamlines_data)
    
    # Create connectivity array for the filtered streamlines
    filtered_lines = []
    point_offset = 0
    for i, streamline_points in enumerate(filtered_streamlines_data):
        n_points = len(streamline_points)
        filtered_lines.append(n_points)
        filtered_lines.extend(range(point_offset, point_offset + n_points))
        point_offset += n_points
    
    # Create new PolyData object
    filtered_streamlines = pv.PolyData()
    filtered_streamlines.points = filtered_points
    filtered_streamlines.lines = np.array(filtered_lines)
    
    # Add classification data
    point_classifications = []
    for i, classification in enumerate(filtered_classifications):
        n_points = len(filtered_streamlines_data[i])
        point_classifications.extend([classification] * n_points)
    
    filtered_streamlines["streamline_set"] = np.array(point_classifications, dtype=np.float32)
    
    return filtered_streamlines, filtered_classifications

# Filter and classify streamlines
filtered_streamlines, classifications = filter_streamlines_by_max_density(streamlines, grid)

if filtered_streamlines is not None:
    # Convert filtered streamlines to tubes
    tubes = filtered_streamlines.tube(radius=0.15)
    
    # Create custom colormap for sets A and C
    colors = ['blue', 'red']
    custom_cmap = ListedColormap(colors)
    
    # Add the filtered field lines to the visualization
    # Use lighting=False to prevent color changes with viewing angle
    plotter.add_mesh(
        tubes,
        scalars="streamline_set",
        cmap = custom_cmap,
        show_scalar_bar = False,
        lighting = False  # This ensures consistent colors regardless of viewing angle
    )
    
    print(f"Added {len(classifications)} filtered streamlines to visualization")
    set_c_count = sum(classifications)
    set_a_count = len(classifications) - set_c_count
    print(f"Blue streamlines (Set A): {int(set_a_count)}")
    print(f"Red streamlines (Set C): {int(set_c_count)}")
else:
    print("No streamlines to display")

# Add title
#plotter.add_text(
#    "Magnetic fields around dense cores from simulations",
#    font_size=16,
#    position="upper_edge"
#)

# Use the xz-plane view as a starting point
plotter.view_xz()

# Further adjust the view angle
#plotter.camera.azimuth = -145
#plotter.camera.elevation = -20
plotter.camera.zoom(1.5)

# Get the current camera's view angle (field of view)
view_angle = plotter.camera.GetViewAngle()
annotation = f"Viewing Angle: {view_angle:.2f}°"

# Add the text annotation in the upper-left corner
#plotter.add_text(annotation, position="upper_left", font_size=10, color="black")
plotter.show_axes()
plotter.show()

Analyzing 184154 streamline points...
Set A (max density < 1e4): 187 streamlines
Set B (1e4 < max density < 1.5e5): 1567 streamlines - ignored
Set C (max density > 1.5e5): 133 streamlines
Total kept: 320 streamlines
Added 320 filtered streamlines to visualization
Blue streamlines (Set A): 187
Red streamlines (Set C): 133


Widget(value='<iframe src="http://localhost:57053/index.html?ui=P_0x1ec77daa900_0&reconnect=auto" class="pyvis…